# Use Case 2: Cache-Augmented Generation (CAG) 

**The Concept:** 
LLM API calls are expensive, slow, and rate-limited. If User A asks *"How do I install SochDB?"* and User B later asks *"What are the installation steps for SochDB?"*, the LLM should not be queried twice because the *intent* is identical.

**The Architecture:** 
We use SochDB's **Semantic Cache**. Instead of caching exact string matches, we cache the semantic meaning of the question. When a new question arrives, we embed it and check if it is semantically identical (e.g., >90% similarity) to a past question. If it is, we instantly return the cached LLM response.

---

### Step 0: Install Packages & Setup Environment
First, we authenticate with Google Gemini using the `.env` file credentials. If you are running this, ensure your `.env` file is properly configured.

In [5]:
!pip install sochdb openai python-dotenv

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import time

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

CHAT_MODEL = "gemini-3-flash-preview"
EMBEDDING_MODEL = "gemini-embedding-001"

def get_embedding(text):
    """Calls Gemini to create a vector embedding of the text."""
    response = client.embeddings.create(input=[text], model=EMBEDDING_MODEL)
    return response.data[0].embedding

def get_completion(prompt):
    """Calls Gemini to generate a chat response."""
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


You should consider upgrading via the '/Users/sushanth/sochdb_python/venv/bin/python3 -m pip install --upgrade pip' command.


### Step 1: Initialize Database
Open an embedded SochDB instance to act as our semantic caching layer.

In [6]:
from sochdb import Database

# Open an embedded database
db = Database.open("./cag_ai_app_db")

### Step 2: Define the Semantic Caching Wrapper
We write a wrapper function around our Google Gemini interaction. The function hits SochDB *first*. If our new prompt's embedding matches something in the database by ≥ 90%, it instantly returns the cached string. If we miss the cache, it calls Google Gemini and then records that completion into SochDB for the next time.

In [10]:
def ask_llm_with_cache(user_question: str):
    """Answers a question by checking the Semantic Cache first."""
    
    # 1. Embed the incoming question using Gemini Embeddings API
    question_embedding = get_embedding(user_question)
    
    # 2. Check SochDB Semantic Cache
    cached_response = db.cache_get(
        cache_name="production_llm_responses",
        query_embedding=question_embedding,
        threshold=0.75  # 75% cosine similarity required to trigger a cache HIT
    )
    
    if cached_response is not None:
        print(f"⚡ CACHE HIT!")
        return cached_response
        
    # 3. Cache Miss - Call the expensive Gemini API endpoint
    print("🐌 CACHE MISS! Calling the Gemini API...")
    actual_response = get_completion(user_question)
    
    # 4. Save the new response into the Semantic Cache for future users
    db.cache_put(
        cache_name="production_llm_responses",
        key=user_question,               # The exact question asked
        value=actual_response,           # The LLM's response
        embedding=question_embedding,    # The semantic meaning of the question
        ttl_seconds=86400                # Expire the cache in 24 hours
    )
    
    return actual_response

### Step 3: Test the Semantic Cache
We simulate two completely different wordings of the exact same question. The first one will be slow (Calling the API). The second one will be near-instant (Retrieving from local SochDB Cache).

In [11]:
print("==== First API Request (Expect API Generation time) ====")
start_time = time.time()
response_1 = ask_llm_with_cache("Explain concisely what a Large Language Model is.")
print(f"Time taken: {time.time() - start_time:.4f} seconds")
print(f"Output: {response_1}\n")

print("==== Second Semantic Cache Request (Expect < 0.1s turnaround) ====")
start_time = time.time()
# We ask practically the same question but worded differently
response_2 = ask_llm_with_cache("Can you briefly explain what LLMs are?")
print(f"Time taken: {time.time() - start_time:.4f} seconds")
print(f"Output: {response_2}")

==== First API Request (Expect API Generation time) ====
⚡ CACHE HIT!
Time taken: 0.5237 seconds
Output: A **Large Language Model (LLM)** is a type of artificial intelligence trained on vast amounts of text to understand, generate, and manipulate human language.

Key characteristics include:
*   **"Large":** They are trained on massive datasets (petabytes of text) and use billions of "parameters" (internal variables) to recognize complex patterns.
*   **Predictive:** At their core, they work by predicting the most statistically likely next word in a sequence.
*   **Versatile:** Unlike older AI, they can perform a wide range of tasks without specific retraining, such as writing code, summarizing articles, translating languages, and holding creative conversations.

==== Second Semantic Cache Request (Expect < 0.1s turnaround) ====
⚡ CACHE HIT!
Time taken: 0.2552 seconds
Output: A **Large Language Model (LLM)** is a type of artificial intelligence trained on vast amounts of text to unders